# 02 - Modeling

**Purpose:** Build the train/test split, the preprocessing pipeline, and the
first models (Logistic Regression baseline, Decision Tree, Random Forest).

**Sections:**
1. Load + rebuild EDA temporal features
2. Column decisions (drop PII, junk, high-cardinality)
3. Stratified 80/20 train/test split
4. Preprocessing pipeline (lives in `src/preprocessing.py`)
5. Baseline model + evaluation

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_DIR = Path("..") / "data" / "raw"
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Load and combine (same as 01_eda.ipynb)
train = pd.read_csv(DATA_DIR / "fraudTrain.csv")
test = pd.read_csv(DATA_DIR / "fraudTest.csv")
df = pd.concat([train, test], ignore_index=True)
del train, test

# Rebuild the EDA temporal features so this notebook is self-contained
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
df["hour"] = df["trans_date_trans_time"].dt.hour
df["day_of_week"] = df["trans_date_trans_time"].dt.dayofweek
df["month"] = df["trans_date_trans_time"].dt.month

print("Loaded:", df.shape)
print("Fraud rate:", round(df["is_fraud"].mean(), 5))

Loaded: (1852394, 26)
Fraud rate: 0.00521


## 2. Column decisions

Dropping:

- **Junk:** `Unnamed: 0` (CSV row index)
- **PII:** `cc_num`, `first`, `last`, `street`, `city`, `zip`, `dob`, `trans_num`
- **High-cardinality:** `merchant` (693), `job` (497)
- **Redundant:** `unix_time` (already have `hour`/`day_of_week`/`month`), `trans_date_trans_time` (already extracted), `lat`/`long`/`merch_lat`/`merch_long` (near-duplicates of each other, no class separation)

Keeping:

- **Numeric:** `amt`, `city_pop`, `hour`, `day_of_week`, `month`
- **Categorical:** `gender`, `category`, `state`
- **Target:** `is_fraud`

In [2]:
DROP_COLS = [
    # junk
    "Unnamed: 0",
    # PII
    "cc_num", "first", "last", "street", "city", "zip", "dob", "trans_num",
    # high-cardinality
    "merchant", "job",
    # redundant / unhelpful
    "unix_time", "trans_date_trans_time",
    "lat", "long", "merch_lat", "merch_long",
]

NUMERIC_COLS = ["amt", "city_pop", "hour", "day_of_week", "month"]
CATEGORICAL_COLS = ["gender", "category", "state"]
TARGET = "is_fraud"

# Sanity check: nothing important is being dropped by accident
assert TARGET not in DROP_COLS
assert all(c in df.columns for c in DROP_COLS)
assert all(c in df.columns for c in NUMERIC_COLS + CATEGORICAL_COLS)

df_model = df.drop(columns=DROP_COLS).copy()

print("Kept columns:", list(df_model.columns))
print("Shape after drop:", df_model.shape)
print()
print(df_model.head(3))

Kept columns: ['category', 'amt', 'gender', 'state', 'city_pop', 'is_fraud', 'hour', 'day_of_week', 'month']
Shape after drop: (1852394, 9)

        category     amt gender state  city_pop  is_fraud  hour  day_of_week  month
0       misc_net    4.97      F    NC      3495         0     0            1      1
1    grocery_pos  107.23      F    WA       149         0     0            1      1
2  entertainment  220.11      M    ID      4154         0     0            1      1


## 3. Train/test split

Stratified 80/20 split on `is_fraud` so the ~0.52% fraud rate is preserved in both halves.

`random_state=42` for reproducibility. The test set is locked and never touched again until the final evaluation.

In [3]:
X = df_model.drop(columns=TARGET)
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print()
print("Train fraud rate:", round(y_train.mean(), 5), f"({y_train.sum():,} fraud)")
print("Test  fraud rate:", round(y_test.mean(), 5),  f"({y_test.sum():,} fraud)")
print()
print("Split preserved the imbalance ratio:", np.isclose(y_train.mean(), y_test.mean(), atol=1e-5))

Train: (1481915, 8)  Test: (370479, 8)

Train fraud rate: 0.00521 (7,721 fraud)
Test  fraud rate: 0.00521 (1,930 fraud)

Split preserved the imbalance ratio: True


### Train/test split - observation

Output looks exactly like it should:

- **Train:** 1,481,915 rows (80%) x 8 features
- **Test:** 370,479 rows (20%) x 8 features
- Fraud counts: 7,721 in train, 1,930 in test. Added together that's 9,651 - which is the exact total fraud count from the full dataset. Nothing lost, nothing double counted.
- **Both splits show a fraud rate of 0.00521** - identical to 5 decimal places. So `stratify=y` did its job. Without it, a random 20% slice could have ended up with meaningfully more or fewer fraud cases, which would quietly skew every downstream comparison.

The one thing I want to remember going forward: **`X_test` and `y_test` are now frozen.** I won't touch them until Step 12 (final evaluation). Every experiment from here on uses `X_train`/`y_train`, and for validation during tuning I'll carve out a slice of the train set inside cross-validation rather than peeking at test. That's the whole point of a held-out set - if I look at it to make decisions, it stops being held out and the final numbers become optimistic.

In [4]:
# Pre-flight check - is X_train actually ready for preprocessing?
print("X_train dtypes:")
print(X_train.dtypes.to_string())
print()
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test: ", X_test.isnull().sum().sum())
print()
print("Categorical cardinalities (on train):")
for c in CATEGORICAL_COLS:
    print(f"  {c:10s}: {X_train[c].nunique():>4} unique values")
print()
print("Numeric ranges (train):")
print(X_train[NUMERIC_COLS].describe().T[["min", "mean", "max"]].round(3).to_string())

X_train dtypes:
category           str
amt            float64
gender             str
state              str
city_pop         int64
hour             int32
day_of_week      int32
month            int32

Missing values in X_train: 0
Missing values in X_test:  0

Categorical cardinalities (on train):
  gender    :    2 unique values
  category  :   14 unique values
  state     :   51 unique values

Numeric ranges (train):
              min       mean         max
amt           1.0     70.022    27390.12
city_pop     23.0  88640.689  2906700.00
hour          0.0     12.808       23.00
day_of_week   0.0      2.968        6.00
month         1.0      7.154       12.00


### Pre-flight check - observation

Everything checks out:

- **Zero missing values** in both train and test. That matches Step 4's finding - this dataset was generated clean, no imputation needed anywhere in the pipeline.
- **Cardinalities are all manageable:**
  - `gender` - 2 (one-hot gives 2 columns)
  - `category` - 14 (one-hot gives 14)
  - `state` - 51 (one-hot gives 51)
  - Total one-hot expansion: 67 columns. Combined with the 5 numerics that's 72 features going into the models. That's totally fine - no dimensionality explosion, no need for target encoding or hashing.
- **Numeric ranges are sane.** `amt` maxes at 27,390 (same outlier we saw in EDA, not a bug), `city_pop` goes up to 2.9M (NYC), `hour` is 0-23, `day_of_week` is 0-6, `month` is 1-12. Nothing out of bounds.

The one thing to keep in mind: `amt`'s distribution is heavily right-skewed (max ~27k, median ~47). Tree models don't care about that, but Logistic Regression will be influenced by those large values even after scaling - a single $27k transaction can dominate a linear decision boundary. That's why I'll apply `log1p` to `amt` **only on the Logistic Regression path**, not on the tree paths. Trees are scale-invariant, so the transform is unnecessary noise for them.

Also worth noting for later: these are dtype-aware. `hour`, `day_of_week`, `month` came through as `int32` - that's fine, `StandardScaler` and tree models both handle ints. If I were to write the pipeline in a more production-ready way I'd cast them to float, but for this project int is fine and scikit-learn won't complain.

So - the split is clean, the column set is final, and the pipeline can now be built on top of it. Next step: write `src/preprocessing.py`.

## 4. Preprocessing pipeline

The pipeline lives in `src/preprocessing.py` so it can be reused across notebooks and (later) inside model scripts. Two variants:

- `build_preprocessor("tree")` - one-hot cats + raw numerics. Trees don't care about scale or skew.
- `build_preprocessor("lr")` - one-hot cats + `log1p(amt)` + `StandardScaler` on all numerics. LR cares about both.

Fitting rule: **fit on `X_train` only.** Then apply the already-fitted transformer to `X_test`. Fitting on the whole dataset would let information from the test set leak into the training process, and every metric after that would be optimistic.

In [5]:
import sys
sys.path.append("..")   # so `from src.preprocessing import ...` works from notebooks/

from src.preprocessing import build_preprocessor

# Fit the tree variant on the training set
pre_tree = build_preprocessor("tree")
X_train_tree = pre_tree.fit_transform(X_train)
X_test_tree  = pre_tree.transform(X_test)

# And the LR variant (fit separately - its internal scaler must also see only training data)
pre_lr = build_preprocessor("lr")
X_train_lr = pre_lr.fit_transform(X_train)
X_test_lr  = pre_lr.transform(X_test)

print("Tree  - train:", X_train_tree.shape, " test:", X_test_tree.shape)
print("LR    - train:", X_train_lr.shape,   " test:", X_test_lr.shape)
print()
print("Sanity checks:")
print("  No NaN in tree train:", not np.isnan(X_train_tree).any())
print("  No NaN in tree test: ", not np.isnan(X_test_tree).any())
print("  No NaN in LR train:  ", not np.isnan(X_train_lr).any())
print("  No NaN in LR test:   ", not np.isnan(X_test_lr).any())
print()
print("Feature names (first 20):")
print(pre_tree.get_feature_names_out()[:20])

Tree  - train: (1481915, 72)  test: (370479, 72)
LR    - train: (1481915, 72)  test: (370479, 72)

Sanity checks:
  No NaN in tree train: True
  No NaN in tree test:  True
  No NaN in LR train:   True
  No NaN in LR test:    True

Feature names (first 20):
['gender_F' 'gender_M' 'category_entertainment' 'category_food_dining'
 'category_gas_transport' 'category_grocery_net' 'category_grocery_pos'
 'category_health_fitness' 'category_home' 'category_kids_pets'
 'category_misc_net' 'category_misc_pos' 'category_personal_care'
 'category_shopping_net' 'category_shopping_pos' 'category_travel'
 'state_AK' 'state_AL' 'state_AR' 'state_AZ']


### Preprocessing pipeline - observation

Everything came out the way it should:

- **Both variants produced 72 features** - 67 from one-hot encoding (`gender` × 2, `category` × 14, `state` × 51) plus 5 numerics. Same shape for tree and LR because the LR variant transforms the numerics in place (log + scale), it doesn't add or remove columns.
- **Zero NaNs** in either output, on either split. That's the bare minimum for scikit-learn to accept the data, so it needed checking.
- **Feature names are clean** - `gender_F`, `category_grocery_pos`, `state_AK` etc. No `cat__` or `num__` prefixes, because I set `verbose_feature_names_out=False` in the pipeline. That'll make it much easier to read feature importances later.
- **Train and test line up row-for-row with what came out of the split** - 1,481,915 and 370,479. No rows lost during transform. Also means `.fit_transform` on train and `.transform` on test are behaving consistently, which is the whole point of fitting on train only.

One thing I want to remember: **`pre_tree` and `pre_lr` are two separate fitted objects.** They each learned their internal state (one-hot category list, LR scaler means/stds, log baseline) from `X_train` only. The `X_test` transform used those already-fitted transformers - it never re-fit. That's the entire anti-leakage discipline, and it's easy to break by accidentally calling `fit_transform` on the test set. I didn't, but it's the kind of mistake that would silently inflate every number downstream.

So the pipeline is done. From here on, every model gets fed `X_train_tree` / `X_test_tree` (for tree models) or `X_train_lr` / `X_test_lr` (for LR). Step 6 is essentially complete.